<a href="https://colab.research.google.com/github/hannahheathomics/ROI-Selection---Damaged-Muscle-Fibers/blob/Organizing/Claude_ROI_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Installs and Imports

In [ ]:
pip install Pillow numpy tifffile scipy

In [ ]:
import argparse
import sys
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw
import tifffile
from scipy.ndimage import label

In [ ]:
# Import pngs
from google.colab import files
from pathlib import Path

uploaded = files.upload()
img_paths = [Path(name) for name in uploaded.keys()]

# Set img_path to the first uploaded file (or loop later)
img_path = img_paths[0]
print(f"Using: {img_path}")

Saving 25R ICA Group 1.PNG to 25R ICA Group 1.PNG
Saving 35B ICA Group 4.PNG to 35B ICA Group 4.PNG
Saving 3B ICA Group 6.PNG to 3B ICA Group 6.PNG
Saving 4N ICA Group 4.PNG to 4N ICA Group 4.PNG
Saving 19B ICA Group 3.PNG to 19B ICA Group 3.PNG
Saving 16N ICA Group 2.PNG to 16N ICA Group 2.PNG
Saving 1R ICA Group 3.PNG to 1R ICA Group 3.PNG
Saving 9R ICA Group 6.PNG to 9R ICA Group 6.PNG
Saving 6L ICA Group 4.PNG to 6L ICA Group 4.PNG
Saving 33R ICA Group 2.PNG to 33R ICA Group 2.PNG
Saving 32N ICA Group 1.PNG to 32N ICA Group 1.PNG
Saving 24N ICA Group 4.PNG to 24N ICA Group 4.PNG
Saving 20N ICA Group 5.PNG to 20N ICA Group 5.PNG
Saving 26L ICA Group 1.PNG to 26L ICA Group 1.PNG
Saving 11B ICA Group 2.PNG to 11B ICA Group 2.PNG
Saving 7B ICA Group 3.PNG to 7B ICA Group 3.PNG
Saving 14L ICA Group 5.PNG to 14L ICA Group 5.PNG
Using: 25R ICA Group 1.PNG


## ROI selection

In [ ]:
#Tissue segmentation

def get_main_tissue_mask(green: np.ndarray, blue: np.ndarray, threshold: int = 15) -> np.ndarray:
  """Return a binary mask of the largest connected tissue region."""
  combined = (green > threshold) | (blue > threshold)
  labeled, _ = label(combined)
  sizes = np.bincount(labeled.ravel())
  sizes[0] = 0  # ignore background
  largest_label = sizes.argmax()
  return labeled == largest_label

In [ ]:
#ROI scoring

def find_best_roi(green: np.ndarray,
                  tissue: np.ndarray,
                  sq: int,
                  stride: int,
                  min_fill: float) -> tuple:
  """
  Slide a sq×sq window over the image and return the (y, x, score) of the
  patch with the highest mean laminin signal within tissue pixels.
  Score = mean_green_in_tissue × tissue_fill_fraction
  """
  H, W = green.shape
  g = green.astype(float) / 255.0
  best_score = -1.0
  best_y, best_x = 0, 0
  for y in range(0, H - sq + 1, stride):
    for x in range(0, W - sq + 1, stride):
      patch_tissue = tissue[y:y + sq, x:x + sq]
      frac = patch_tissue.mean()
      if frac < min_fill:
        continue
      mean_g = g[y:y + sq, x:x + sq][patch_tissue].mean()
      score = mean_g * frac
      if score > best_score:
        best_score = score
        best_y, best_x = y, x
    return best_y, best_x, best_score

In [ ]:
# Per-image pipeline

def process_image(img_path: Path,
                  out_dir: Path,
                  sq_override,
                  stride: int,
                  min_fill: float,
                  edge_margin: float) -> None:
    print("\nProcessing: " + img_path.name)

In [ ]:
#Process images
for img_path in img_paths:
    print("\nProcessing: " + img_path.name)

    # Load image
    img   = Image.open(img_path).convert("RGBA")
    arr   = np.array(img)
    green = arr[:, :, 1]
    blue  = arr[:, :, 2]
    H, W  = green.shape

    # Tissue mask
    tissue = get_main_tissue_mask(green, blue)

    # Bounding box with edge margin
    rows = np.any(tissue, axis=1)
    cols = np.any(tissue, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    edge_margin = 0.15
    row_margin  = int((rmax - rmin) * edge_margin)
    col_margin  = int((cmax - cmin) * edge_margin)
    inner_rmin  = rmin + row_margin
    inner_rmax  = rmax - row_margin
    inner_cmin  = cmin + col_margin
    inner_cmax  = cmax - col_margin

    sq = min(450, (inner_rmax - inner_rmin), (inner_cmax - inner_cmin))

    # ── Adaptive min_fill ────────────────────────────────────────────────────
    # Calculate actual tissue coverage in the interior search zone
    # and set min_fill to 70% of that — so the threshold scales to each image
    interior_tissue  = tissue[inner_rmin:inner_rmax, inner_cmin:inner_cmax]
    actual_fill      = interior_tissue.mean()
    min_fill         = round(actual_fill * 0.70, 3)
    min_fill         = max(0.20, min(min_fill, 0.60))  # clamp between 0.20 and 0.60
    print("  Interior tissue fill: " + f"{actual_fill:.3f}" + "  →  adaptive min_fill: " + str(min_fill))
    # ─────────────────────────────────────────────────────────────────────────

    sub_green  = green[inner_rmin:inner_rmax, inner_cmin:inner_cmax]
    sub_tissue = tissue[inner_rmin:inner_rmax, inner_cmin:inner_cmax]
    rel_y, rel_x, score = find_best_roi(sub_green, sub_tissue, sq, 15, min_fill)

    if score < 0:
        rel_y = (inner_rmax - inner_rmin) // 2 - sq // 2
        rel_x = (inner_cmax - inner_cmin) // 2 - sq // 2
        print("  Fallback to centre — interior fill too low even after adaptation")

    abs_y = max(0, min(inner_rmin + rel_y, H - sq))
    abs_x = max(0, min(inner_cmin + rel_x, W - sq))
    print("  ROI origin: (" + str(abs_y) + ", " + str(abs_x) + ") score=" + f"{score:.4f}")

    stem    = img_path.stem
    out_dir = img_path.parent
    out_dir.mkdir(parents=True, exist_ok=True)

    laminin_crop = green[abs_y:abs_y + sq, abs_x:abs_x + sq]
    roi_path = out_dir / (stem + "_laminin_ROI.tiff")
    tifffile.imwrite(str(roi_path), laminin_crop)
    print("  Saved ROI : " + str(roi_path))

    rgb  = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(rgb)
    for offset in range(6):
        draw.rectangle(
            [abs_x - offset, abs_y - offset,
             abs_x + sq + offset, abs_y + sq + offset],
            outline=(255, 220, 0)
        )
    qc_path = out_dir / (stem + "_QC_preview.tiff")
    rgb.save(str(qc_path), format="TIFF")
    print("  Saved QC  : " + str(qc_path))

print("\nDone.")


Processing: 25R ICA Group 1.PNG
  Interior tissue fill: 0.225  →  adaptive min_fill: 0.2
  ROI origin: (973, 2386) score=0.0618
  Saved ROI : 25R ICA Group 1_laminin_ROI.tiff
  Saved QC  : 25R ICA Group 1_QC_preview.tiff

Processing: 35B ICA Group 4.PNG
  Interior tissue fill: 0.401  →  adaptive min_fill: 0.28
  ROI origin: (845, 1003) score=0.1545
  Saved ROI : 35B ICA Group 4_laminin_ROI.tiff
  Saved QC  : 35B ICA Group 4_QC_preview.tiff

Processing: 3B ICA Group 6.PNG
  Interior tissue fill: 0.610  →  adaptive min_fill: 0.427
  ROI origin: (261, 825) score=0.1933
  Saved ROI : 3B ICA Group 6_laminin_ROI.tiff
  Saved QC  : 3B ICA Group 6_QC_preview.tiff

Processing: 4N ICA Group 4.PNG
  Interior tissue fill: 0.533  →  adaptive min_fill: 0.373
  ROI origin: (530, 629) score=0.1364
  Saved ROI : 4N ICA Group 4_laminin_ROI.tiff
  Saved QC  : 4N ICA Group 4_QC_preview.tiff

Processing: 19B ICA Group 3.PNG
  Interior tissue fill: 0.466  →  adaptive min_fill: 0.327
  ROI origin: (535, 625

In [ ]:
#Generate single QC image
import math
from PIL import Image

# Load all QC previews
qc_images = []
for img_path in img_paths:
    qc_path = img_path.parent / (img_path.stem + "_QC_preview.tiff")
    if qc_path.exists():
        qc_images.append((img_path.stem, Image.open(qc_path).convert("RGB")))
    else:
        print("Missing QC for: " + img_path.stem)

if not qc_images:
    print("No QC images found — run the pipeline first.")
else:
    # Grid layout
    n      = len(qc_images)
    cols   = math.ceil(math.sqrt(n))
    rows   = math.ceil(n / cols)
    thumb  = 512  # thumbnail size per image
    pad    = 20   # padding between images
    label_height = 24

    cell_w = thumb + pad
    cell_h = thumb + pad + label_height
    grid_w = cols * cell_w + pad
    grid_h = rows * cell_h + pad

    grid = Image.new("RGB", (grid_w, grid_h), color=(30, 30, 30))

    from PIL import ImageDraw, ImageFont
    draw = ImageDraw.Draw(grid)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 16)
    except:
        font = ImageFont.load_default()

    for idx, (stem, img) in enumerate(qc_images):
        row = idx // cols
        col = idx % cols
        x = pad + col * cell_w
        y = pad + row * cell_h

        thumb_img = img.copy()
        thumb_img.thumbnail((thumb, thumb), Image.LANCZOS)
        grid.paste(thumb_img, (x, y))
        draw.text((x, y + thumb + 4), stem, fill=(220, 220, 220), font=font)

    # Save and download
    out_path = img_paths[0].parent / "QC_compilation.tiff"
    grid.save(str(out_path), format="TIFF")
    print("Saved compilation: " + str(out_path))

    from google.colab import files
    files.download(str(out_path))


Saved compilation: QC_compilation.tiff


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## QC file verification

Check the QC file.

There will likely be a few files where the selection is wrong. Below are code snippets to fix it.

###**Problem: Selected area isn't muscle fiber**

Solution below


In [ ]:
# ── Manual re-run with nerve/artefact exclusion ───────────────────────────────
rerun_filename    = "19B ICA Group 3.PNG"   # change to your problem file
rerun_min_fill    = 0.35
rerun_stride      = 10
rerun_sq          = None
blue_exclude_thresh = 30             # pixels with blue > this are masked out (nerve/connective tissue)

# ── Do not edit below ─────────────────────────────────────────────────────────
img_path = next((p for p in img_paths if p.name == rerun_filename), None)

if img_path is None:
    print("File not found: " + rerun_filename)
    print("Available: " + str([p.name for p in img_paths]))
else:
    img   = Image.open(img_path).convert("RGBA")
    arr   = np.array(img)
    green = arr[:, :, 1]
    blue  = arr[:, :, 2]
    H, W  = green.shape

    # Build tissue mask but exclude high-blue pixels (nerve/connective tissue)
    tissue = get_main_tissue_mask(green, blue)
    blue_mask = blue > blue_exclude_thresh
    tissue_clean = tissue & ~blue_mask

    # Bounding box on cleaned mask
    rows = np.any(tissue_clean, axis=1)
    cols = np.any(tissue_clean, axis=0)

    if not rows.any() or not cols.any():
        print("Clean mask is empty — try increasing blue_exclude_thresh")
    else:
        rmin, rmax = np.where(rows)[0][[0, -1]]
        cmin, cmax = np.where(cols)[0][[0, -1]]

        # No edge margin needed — blue exclusion handles the fragment
        inner_rmin, inner_rmax = rmin, rmax
        inner_cmin, inner_cmax = cmin, cmax

        sq = rerun_sq if rerun_sq else min(450, (inner_rmax - inner_rmin), (inner_cmax - inner_cmin))

        sub_green  = green[inner_rmin:inner_rmax, inner_cmin:inner_cmax]
        sub_tissue = tissue_clean[inner_rmin:inner_rmax, inner_cmin:inner_cmax]
        rel_y, rel_x, score = find_best_roi(sub_green, sub_tissue, sq, rerun_stride, rerun_min_fill)

        if score < 0:
            rel_y = (inner_rmax - inner_rmin) // 2 - sq // 2
            rel_x = (inner_cmax - inner_cmin) // 2 - sq // 2
            print("  Fallback to centre — try lowering blue_exclude_thresh or min_fill")

        abs_y = max(0, min(inner_rmin + rel_y, H - sq))
        abs_x = max(0, min(inner_cmin + rel_x, W - sq))
        print("  ROI origin: (" + str(abs_y) + ", " + str(abs_x) + ") score=" + f"{score:.4f}")

        stem    = img_path.stem
        out_dir = img_path.parent
        out_dir.mkdir(parents=True, exist_ok=True)

        laminin_crop = green[abs_y:abs_y + sq, abs_x:abs_x + sq]
        tifffile.imwrite(str(out_dir / (stem + "_laminin_ROI.tiff")), laminin_crop)
        print("  Saved ROI : " + stem + "_laminin_ROI.tiff")

        rgb  = Image.open(img_path).convert("RGB")
        draw = ImageDraw.Draw(rgb)
        for offset in range(6):
            draw.rectangle(
                [abs_x - offset, abs_y - offset,
                 abs_x + sq + offset, abs_y + sq + offset],
                outline=(255, 220, 0)
            )
        rgb.save(str(out_dir / (stem + "_QC_preview.tiff")), format="TIFF")
        print("  Saved QC  : " + stem + "_QC_preview.tiff")
        print("\nDone. Re-run QC compilation to update the grid.")

  Regions found: 88278  — keeping largest, excluding 88277 fragment(s)
  Interior tissue fill: 0.107  →  adaptive min_fill: 0.2
  ROI origin: (326, 641) score=0.1156
  Saved ROI : 19B ICA Group 3_laminin_ROI.tiff
  Saved QC  : 19B ICA Group 3_QC_preview.tiff

Done. Re-run QC compilation to update the grid.


###**Problem: Selected area includes too much slide background**

Solution below


In [ ]:
# ── Manual re-run for background/edge problem ─────────────────────────────────
rerun_filename      = "14L ICA Group 5.PNG"  # change to your problem file
rerun_edge_margin   = 0.05                   # Reduce number to reduce amount of slide background
rerun_stride        = 10
rerun_sq            = None
tissue_threshold    = 45                     # raise this to ignore dim/sparse tissue pixels

# ── Do not edit below ─────────────────────────────────────────────────────────
img_path = next((p for p in img_paths if p.name == rerun_filename), None)

if img_path is None:
    print("File not found: " + rerun_filename)
    print("Available: " + str([p.name for p in img_paths]))
else:
    img   = Image.open(img_path).convert("RGBA")
    arr   = np.array(img)
    green = arr[:, :, 1]
    blue  = arr[:, :, 2]
    H, W  = green.shape

    # Stricter tissue mask — only count brighter pixels as tissue
    from scipy.ndimage import label as nd_label
    combined = (green > tissue_threshold) | (blue > tissue_threshold)
    labeled, _ = nd_label(combined)
    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0
    tissue = labeled == sizes.argmax()

    # Bounding box with edge margin to focus on dense interior
    rows = np.any(tissue, axis=1)
    cols = np.any(tissue, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]

    row_margin = int((rmax - rmin) * rerun_edge_margin)
    col_margin = int((cmax - cmin) * rerun_edge_margin)
    inner_rmin = rmin + row_margin
    inner_rmax = rmax - row_margin
    inner_cmin = cmin + col_margin
    inner_cmax = cmax - col_margin

    sq = rerun_sq if rerun_sq else min(450, (inner_rmax - inner_rmin), (inner_cmax - inner_cmin))

    # Adaptive min_fill — same logic as main loop
    interior_tissue = tissue[inner_rmin:inner_rmax, inner_cmin:inner_cmax]
    actual_fill     = interior_tissue.mean()
    min_fill        = round(actual_fill * 0.70, 3)
    min_fill        = max(0.20, min(min_fill, 0.60))
    print("  Interior tissue fill: " + f"{actual_fill:.3f}" + "  →  adaptive min_fill: " + str(min_fill))

    sub_green  = green[inner_rmin:inner_rmax, inner_cmin:inner_cmax]
    sub_tissue = tissue[inner_rmin:inner_rmax, inner_cmin:inner_cmax]
    rel_y, rel_x, score = find_best_roi(sub_green, sub_tissue, sq, rerun_stride, min_fill)

    if score < 0:
        rel_y = (inner_rmax - inner_rmin) // 2 - sq // 2
        rel_x = (inner_cmax - inner_cmin) // 2 - sq // 2
        print("  Fallback to centre — try raising tissue_threshold or edge_margin")

    abs_y = max(0, min(inner_rmin + rel_y, H - sq))
    abs_x = max(0, min(inner_cmin + rel_x, W - sq))
    print("  ROI origin: (" + str(abs_y) + ", " + str(abs_x) + ") score=" + f"{score:.4f}")

    stem    = img_path.stem
    out_dir = img_path.parent
    out_dir.mkdir(parents=True, exist_ok=True)

    laminin_crop = green[abs_y:abs_y + sq, abs_x:abs_x + sq]
    tifffile.imwrite(str(out_dir / (stem + "_laminin_ROI.tiff")), laminin_crop)
    print("  Saved ROI : " + stem + "_laminin_ROI.tiff")

    rgb  = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(rgb)
    for offset in range(6):
        draw.rectangle(
            [abs_x - offset, abs_y - offset,
             abs_x + sq + offset, abs_y + sq + offset],
            outline=(255, 220, 0)
        )
    rgb.save(str(out_dir / (stem + "_QC_preview.tiff")), format="TIFF")
    print("  Saved QC  : " + stem + "_QC_preview.tiff")
    print("\nDone. Re-run QC compilation to update the grid.")


  Interior tissue fill: 0.225  →  adaptive min_fill: 0.2
  ROI origin: (511, 1606) score=0.1516
  Saved ROI : 14L ICA Group 5_laminin_ROI.tiff
  Saved QC  : 14L ICA Group 5_QC_preview.tiff

Done. Re-run QC compilation to update the grid.
